<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.14}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumnos:} & Elizalde Maza Jesús Eduardo 321031686 \\[6pt]
& Ledesma Cuevas Eduardo 321249050 \\[6pt]
& Peredo López Citlalli Abigail  321161022 \\[6pt]
& Santana de la Rosa Monica Guadalupe 321023663 \\[6pt]
\textbf{Fecha de realización:} & 15/03/2026
\end{array}

</center>

# 1. Base de Conocimiento (Prolog)

La base de conocimiento define la ubicación ideal de los productos y las reglas para el diagnóstico y la planificación

# 2. Estados Iniciales y Finales

Para un programa de toma de decisiones, definimos los estados de la siguiente manera:Estado Inicial: El robot se encuentra en la "posición inicial" (centro del triángulo que forman los estantes). Los estantes tienen una distribución de productos que puede o no coincidir con la ideal.Estado Final:
- 1.  El cliente ha recibido el producto solicitado (ej. "Aquí está el refresco").
- 2.  Todos los productos observados durante la tarea han sido reubicados en sus estantes correctos según la base de conocimiento ("Productos Acomodados").

# 3. Integración en Python (Uso de pyswip)
Para conectar esto con Python, se utiliza la librería pyswip. El flujo consiste en que Python maneja la interfaz y el robot, mientras Prolog toma las decisiones lógicas.

In [1]:
# 1. Instalar el motor de SWI-Prolog en el sistema
!apt-get install -y swi-prolog

# 2. Instalar la librería puente para Python
!pip install pyswip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-sendmail-perl libossp-uuid16 libsub-override-perl
  libsys-hostname-long-perl libtool po-debconf swi-prolog-core
  swi-prolog-core-packages swi-prolog-doc swi-prolog-nox swi-prolog-x
Suggested packages:
  dh-make gettext-doc libasprintf-dev libgettextpo-dev uuid libtool-doc
  gcj-jdk libmail-box-perl elpa-ediprolog swi-prolog-java swi-prolog-odbc
  swi-prolog-bdb
The following NEW packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-send

Crear un archivo como base de conocimiento


% Ubicaciones ideales (Creencia inicial del sistema)

ideal(estante_bebidas, [cerveza, refresco]).

ideal(estante_comida, [sopa, cereal]).

ideal(estante_pan, [galletas]).


% Estado actual (Hechos dinámicos que Python actualizará tras la observación)

% Ejemplo de un estado con desorden (Caso 1.2 b)

ubicado_en(estante_bebidas, refresco).

ubicado_en(estante_bebidas, cerveza).

ubicado_en(estante_bebidas, sopa). % Producto desacomodado

ubicado_en(estante_comida, cereal).

ubicado_en(estante_pan, galletas).

% Regla de Diagnóstico: Identifica qué productos no están en su lugar

diagnostico(Producto, EstanteActual, EstanteCorrecto) :-

    ubicado_en(EstanteActual, Producto),

    ideal(EstanteCorrecto, ProductosIdeales),

    member(Producto, ProductosIdeales),

    EstanteActual \= EstanteCorrecto.


% Regla de Planificación: Genera las acciones necesarias

plan(entregar(P)) :- ubicado_en(_, P).

plan(acomodar(P, Destino)) :- diagnostico(P, _, Destino).

In [ ]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl") # Carga la base de conocimiento

def ejecutar_asistente(producto_buscado):
    print(f"Buscando: {producto_buscado}")

    # 1. Consultar a Prolog por el plan de entrega
    plan_entrega = list(prolog.query(f"plan(entregar({producto_buscado}))"))

    # 2. Consultar si hay algo que reacomodar (Diagnóstico)
    objetos_desacomodados = list(prolog.query("plan(acomodar(P, Destino))"))

    if objetos_desacomodados:
        for obj in objetos_desacomodados:
            print(f"Acción: Mover {obj['P']} al {obj['Destino']}")

    if plan_entrega:
        print(f"Acción: Entregar {producto_buscado} al cliente.")
    else:
        print("Error: Producto no encontrado en el sistema.")

ejecutar_asistente("refresco")

Buscando: refresco
Acción: Mover sopa al estante_comida
Acción: Entregar refresco al cliente.


# Escenario del Desafío:

El cliente solicita un refresco. El robot llega al "Estante de Bebidas" y lo encuentra vacío. Al inspeccionar el "Estante de Comida", descubre que allí están la cerveza, la sopa y las galletas, mientras que el refresco y el cereal están en el "Estante de Pan".
**Reto para el programador:**
Implementar una regla en Prolog que permita al robot priorizar: ¿Debe acomodar todo primero o entregar el refresco cuanto antes para satisfacer al cliente?.

Manejar el "Error de Búsqueda": Si el robot va a un estante esperando algo y no lo encuentra, debe disparar un nuevo diagnóstico (re-planificación en tiempo real).

Restricción: El robot solo puede cargar un máximo de dos objetos a la vez. ¿Cómo optimiza el movimiento entre los 3 estantes para minimizar la distancia recorrida?

In [19]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl") # Carga la base de conocimiento

def ejecutar_asistente(producto_buscado):
    print(f"Buscando: {producto_buscado}")

    # 1. Consultar a Prolog por el plan de entrega
    plan_entrega = list(prolog.query(f"plan(entregar({producto_buscado}))"))

    # 2.  Consultar si hay algo que reacomodar (Diagnóstico)
    pares = list(prolog.query("plan(acomodar(P1,D1,P2,D2))"))
    individuales = list(prolog.query("plan(acomodar(P,D))"))

    if pares:
        for obj in pares:
            print(f"Acción: Mover {obj['P1']} al {obj['D1']} y {obj['P2']} al {obj['D2']}")
    elif individuales:
        for obj in individuales:
            print(f"Acción: Mover {obj['P']} al {obj['D']}")

    if plan_entrega:
        print(f"Acción: Entregar {producto_buscado} al cliente.")
    else:
        print("Error: Producto no encontrado en el sistema.")

ejecutar_asistente("refresco")

Buscando: refresco
Acción: Mover sopa al estante_comida
Acción: Entregar refresco al cliente.


% Ubicaciones ideales (Creencia inicial del sistema)
ideal(estante_bebidas, [cerveza, refresco]).
ideal(estante_comida, [sopa, cereal]).
ideal(estante_pan, [galletas]).

% Estado actual (Hechos dinámicos que Python actualizará tras la observación)
% Ejemplo de un estado con desorden (Caso 1.2 b)
ubicado_en(estante_bebidas, refresco).
ubicado_en(estante_bebidas, cerveza).
ubicado_en(estante_bebidas, sopa). % Producto desacomodado
ubicado_en(estante_comida, cereal).
ubicado_en(estante_pan, galletas).

% Regla de Diagnóstico: Identifica qué productos no están en su lugar [cite: 19]
diagnostico(Producto, EstanteActual, EstanteCorrecto) :-
    ubicado_en(EstanteActual, Producto),
    ideal(EstanteCorrecto, ProductosIdeales),
    member(Producto, ProductosIdeales),
    EstanteActual \= EstanteCorrecto.

% Manejar el error de búsqueda
% Caso normal: el producto está en algún estante
buscar_producto(P, Estante) :- ubicado_en(Estante, P).

% Si no está donde debería, replanifica
buscar_producto(P, EstanteReal) :-
    ideal(EstanteEsperado, Lista),
    member(P, Lista),
    \+ ubicado_en(EstanteEsperado, P),
    ubicado_en(EstanteReal, P).

% Regla de Planificación: Genera las acciones necesarias [cite: 24]
% Prioridad 1: satisfacer al cliente
plan(entregar(P)) :- ubicado_en(_, P), !.

% Prioridad 2: mover dos productos mal ubicados en un viaje
plan(acomodar(P1, D1, P2, D2)) :- acomodar(P1, D1, P2, D2), !.

% Prioridad 3: mover uno si solo hay uno
plan(acomodar(P, D)) :- acomodar(P, D).

% Minimizar la distancia recorrida
% El robot solo puede cargar 2 objetos
capacidad_robot(2).

% Acomodar dos productos con su destino
acomodar(P1, D1, P2, D2) :-
    diagnostico(P1, _, D1),
    diagnostico(P2, _, D2),
    P1 \= P2.

% Acomodar uno si solo hay uno
acomodar(P, D) :- diagnostico(P, _, D).

<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>